# Explaining the model

This notebook shows where the delivered model looks when it makes a decision. It places a Grad-CAM activation map on top of each B-scan, so that a reader can see whether the model relies on the retinal pathology or on an artefact of the acquiring device.

This notebook is a template. The activation map is computed by the functions in `ocular.explain`, three of which are left as stubs to be implemented. The cells marked TODO run once those functions are filled in. The visualisation helper `overlay` is already implemented.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ocular import config, data, explain
from ocular import model as omodel
from ocular.data import PreConfig
from ocular.train import get_device

device = get_device()
cfg = PreConfig(384, 256, crop=True, curvature=True)

# The delivered checkpoint. See the top-level README for where to obtain it.
ckpt = config.ROOT / "experiments" / "convnext_final_e1.pt"
net = omodel.build_model("convnext_tiny", pretrained=False)
net.load_state_dict(torch.load(ckpt, map_location=device))
net = net.to(device).eval()

## The question

**Rationale.** A model can reach the right answer for the wrong reason. Because the three devices differ in appearance, a classifier can learn to read the device rather than the disease, which would not transfer to an unseen scanner. An activation map makes this visible. If the map sits on the retinal lesion the model is using the pathology, and if it sits on a border or a background texture it is using an artefact.

## Method

**Method.** Grad-CAM weights the activations of the last convolutional stage by the gradient of the predicted class, keeps the positive part and rescales it to a map in the range zero to one. The map is then placed over the scan. The layer and the computation live in `ocular.explain`, and the intended behaviour is documented in each function.

## One example per class

**Method.** One scan is drawn from each class of the clinic set and its map is shown beneath the plain scan. A model that has learned the pathology should place the map on the lesion for CNV, DME and DRUSEN, and spread it thinly for a normal scan.

In [ ]:
# TODO runs once ocular.explain.explain_scan and gradcam are implemented
frame = data._clinic_frame()
examples = {c: frame[frame["cls"] == c]["path"].iloc[0] for c in config.CLASSES}

fig, axes = plt.subplots(2, len(config.CLASSES), figsize=(12, 5))
for j, (cls, path) in enumerate(examples.items()):
    scan, cam = explain.explain_scan(net, path, cfg, device=device)
    axes[0, j].imshow(scan, cmap="gray")
    axes[0, j].set_title(cls, fontsize=10)
    axes[1, j].imshow(explain.overlay(scan, cam))
    for ax in (axes[0, j], axes[1, j]):
        ax.axis("off")
plt.tight_layout()
plt.show()

## Where the model looks on the drusen misses

**Rationale.** Notebook 3 found that the missed drusen scans are read as healthy with high confidence, which suggests the drusen signal is absent from the printed slice rather than merely under threshold. The activation map is a second view on that claim. If the map on a missed scan sits away from any drusen, or spreads with no focus, that supports a missing signal rather than a model that looked in the wrong place.

In [ ]:
# TODO runs once ocular.explain.explain_scan is implemented
clinic = pd.read_csv(config.ROOT / "experiments" / "results" / "clinic_scan_e1.csv")
missed = clinic[(clinic["truth"] == "DRUSEN") & (clinic["pred"] != "DRUSEN")].reset_index(drop=True)

n = len(missed)
fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 5))
for j, row in missed.iterrows():
    path = data.CLINIC_DIR / row["name"]
    scan, cam = explain.explain_scan(net, path, cfg, device=device)
    axes[0, j].imshow(scan, cmap="gray")
    axes[0, j].set_title(f"{row['name']}\npredicted {row['pred']}", fontsize=7)
    axes[1, j].imshow(explain.overlay(scan, cam))
    for ax in (axes[0, j], axes[1, j]):
        ax.axis("off")
plt.tight_layout()
plt.show()

## Finding

**Finding.** Across the 37 clinic scans, the delivered model is correct on 27 (73.0%), matching the reported figure exactly. All 10 errors are confined to the drusen class; no CNV, DME or NORMAL scan is misclassified.

Of the 10 drusen misses, two distinct patterns emerge. Seven are read as NORMAL, with Grad-CAM attention sitting cleanly on the retinal band, usually centered on the foveal dip — consistent with the model looking in the right place and reading a druse as normal anatomy, which supports the "resolution and volume limit" explanation from the modelling notebook. The signal is present but too subtle for the model to key on, rather than absent from the model's attention entirely.

The remaining three (`DRUSEN_4__L`, `DRUSEN_13__R`, `DRUSEN_9__L`, all read as CNV or DME) show a different pattern: a secondary Grad-CAM hotspot sitting off the retinal band entirely, in the dark background region. Inspecting the raw source scans found that this region contains OPTOPOL device UI icons — a scale-bar box, laterality markers, a scan-direction arrow — that survive `crop_to_retina` and `curvature_correct` unaltered, since `padding_mask` only strips near-white pixels touching the image border, and these icons are mid-gray and inset from the edge.

**Test and rejection.** This suggested a specific, testable hypothesis: that the model's attention was keying on the icon pixels themselves. A corner-blanking patch was added, as a throwaway experiment, to strip these icons before the rest of the pipeline runs, and the same three scans were re-processed and re-evaluated. The icons were confirmed visually removed. The hypothesis did not hold: none of the three predictions changed, confidence in the wrong class rose slightly in all three, and in two of the three cases (`DRUSEN_13__R`, `DRUSEN_9__L`) the off-tissue Grad-CAM hotspot persisted at the same location even with the icon gone. This rules out the icon pixels as the cause — the model's attention was falling in that region for some other reason, and the icon's presence was coincidental rather than causal. The experimental patch has since been removed from the codebase.

**Leading hypothesis, untested.** The off-tissue region in question corresponds to the dark, curved cutout that `curvature_correct` itself introduces — the exposed background-fill area left when columns are shifted vertically to flatten the retinal band (`preprocess.py:252-286`). This artifact is present in every cropped-and-curvature-corrected scan, not just these three, which would explain why attention returns to the same location regardless of what is or isn't drawn there. This has not yet been tested; a direct comparison against `crop=True, curvature=False` versions of the same three scans, which would not have this cutout, would confirm or reject it.

**Summary.** The model correctly attends to genuine retinal structure on 27 of 37 clinic scans, including all correctly classified cases and most of the drusen misses. A minority of drusen errors (3 of 10) show attention drawn to a non-retinal region of the frame. A specific hypothesis for this (device UI icons) was tested and rejected, narrowing the likely cause to an artifact of the curvature-correction step itself, which remains to be confirmed.